# Auditoria dos CSVs de trafego (ANTT)

Primeira olhada nos dados brutos: schema, encoding, tipos.
Fase F1/F2 do plano de tarefas.

In [ ]:
import pandas as pd
from pathlib import Path

print('pandas', pd.__version__)

In [ ]:
ARQ = Path("../data/raw/volume_pedagio_2024.csv")

df = pd.read_csv(ARQ, sep=';', encoding='latin-1', decimal=',', nrows=100);

print(df.shape)
print(df.dtypes)

df.head()

In [ ]:
for ano in [2024, 2025, 2026]:
    arq = Path(f"../data/raw/volume_pedagio_{ano}.csv")
    df = pd.read_csv(arq, sep=';', decimal = ',', encoding = 'latin-1', nrows=100_000);
    print("Ano:", ano, " | Shape:", df.shape)
    print(df.dtypes)
    print("-" * 40)

In [ ]:
# os conjuntos de valores de sentido, tipo_cobranca e tipo_de_veiculo são iguais nos 3 anos?

from pathlib import Path
import pandas as pd

COLS = ["sentido", "tipo_cobranca", "tipo_de_veiculo"]

for ano in [2024, 2025, 2026]:
    arq = Path(f"../data/raw/volume_pedagio_{ano}.csv")
    df = pd.read_csv(arq, sep=";", encoding="latin-1", usecols=COLS)
    print("=== Ano", ano, "| linhas:", len(df))
    for c in COLS:
        print("  ", c, "->", sorted(df[c].unique()))


In [ ]:
# como foi encontrado um tipo de veículo único em 2024 que não estava nos outros anos, vamos verificar se o valor ou quantidade de ocorrências desse tipo de veículo é relevante para o modelo. 
# Se for irrelevante, podemos simplesmente removê-lo do dataset.

ARQ = Path("../data/raw/volume_pedagio_2024.csv")

df = pd.read_csv(
    ARQ,
    sep=";",
    encoding="latin-1",
    decimal=",",
    usecols=["tipo_de_veiculo", "volume_total"], 
)

mascara = df["tipo_de_veiculo"] == "Veículo Pequeno"
print("linhas de Veiculo Pequeno:", mascara.sum())

vol_pequeno = df.loc[mascara, "volume_total"].sum()
vol_total = df["volume_total"].sum()

print("Veiculo Pequeno:", vol_pequeno)
print("Total 2024 .....:", vol_total)
print("Participacao ...:", vol_pequeno / vol_total)




In [ ]:
# unificando arquivos de 2024, 2025 e 2026 em um único dataset

from pathlib import Path
import pandas as pd

frames = []
for ano in [2024, 2025, 2026]:
    arq = Path(f"../data/raw/volume_pedagio_{ano}.csv")
    df = pd.read_csv(arq, sep=";", encoding="latin-1", decimal=",")
    df["ano_origem"] = ano
    frames.append(df)
    print("lido", ano, "->", df.shape)

dados = pd.concat(frames, ignore_index=True)
print("TOTAL:", dados.shape)

In [ ]:
# Destrinchar o categoria_eixo para verificar se há alguma inconsistência nos dados

for ano in [2024, 2025, 2026]:
    arq = Path(f"../data/raw/volume_pedagio_{ano}.csv")
    df = pd.read_csv(arq, sep=";", encoding="latin-1", usecols=["categoria_eixo"], dtype="str")
    valores = df["categoria_eixo"]

    # (1) tenta converter para numero; o que nao for numero vira NaN (vazio)
    numero = pd.to_numeric(valores, errors="coerce")

    # (2) TODO: use a mascara numero.isna() para listar os valores suspeitos (unicos)
    suspeitos = valores[____].unique()

    print(ano, "->", suspeitos)